# M2 - Agentic AI - Improving SQL Generation with Reflection

## 1. Introduction

### 1.1. Lab overview
In this lab you explore how the **reflection pattern** improves an agentic workflow that converts natural-language questions into SQL. The agent spots issues in its own outputs, refines them, and improves its response before giving a final answer.

### 🎯 1.2 Learning outcome
You will code a reflection step into the workflow, so the agent:
- Reviews its own intermediate results (draft SQL or tool outputs).
- Identifies errors or gaps.
- Checks its responses and tool use.
- Refines the output before submitting the final answer.

> **Before you start:** copy `.env.example` (in the `ReflectDesinPatternDemo` folder) to `.env` and add your `OPENAI_API_KEY`.

## 2. Setup: Initialize Environment and Client
- `json` for handling structured data.
- `pandas` for working with tabular data.
- `dotenv` to load environment variables (e.g., API keys).
- `utils.py`: helper functions to build the database, run SQL and format outputs.

In [ ]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv

_ = load_dotenv()

### 2.1 Getting started with AISuite
[aisuite](https://github.com/andrewyng/aisuite) gives a single, unified way to call LLMs from different providers, using `provider:model` names such as `openai:gpt-4.1`.

In [ ]:
import aisuite as ai

client = ai.Client()

### 2.2. Set Up the Database
Create a local SQLite database called `products.db`, automatically filled with randomly generated product data.

In [ ]:
utils.create_transactions_db()

You can inspect the table schema by executing the cell below.

In [ ]:
utils.print_html(utils.get_schema('products.db'))

In this table, every row represents an **event** (insert, restock, sale, or price update). Stock levels, sales and pricing trends are all derived from these events.

🔎 **Schema overview:**
- `id` → unique event ID (autoincrement).
- `product_id`, `product_name`, `brand`, `category`, `color` → identify the product.
- `action` → type of event (`insert`, `restock`, `sale`, `price_update`).
- `qty_delta` → stock change (+ for insert/restock, – for sale, 0 for price updates).
- `unit_price` → price at that moment (NULL for restock).
- `notes` → optional description of the event.
- `ts` → timestamp when the event was logged.

## 3. Build a SQL generator

### 3.1. Use an LLM to Query a Database
Give the LLM your question and the database schema; it generates the SQL query that answers your question.

In [ ]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

Run the cell below to see how `generate_sql` creates the first version (V1) of an SQL query from a plain-English question.

In [ ]:
# Example usage of generate_sql

# We provide the schema as a string
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

# We ask a question about the data in natural language
question = "Which color of product has the highest total sales?"

utils.print_html(question, title="User Question")

# Generate the SQL query using the specified model
sql_V1 = generate_sql(question, schema, model="openai:gpt-4.1")

# Display the generated SQL query
utils.print_html(sql_V1, title="SQL Query V1")

#### 3.1.1. Query Execution
Execute V1 and inspect its results to verify whether the query actually retrieves what the question asks for.
- `utils.execute_sql(...)` runs the SQL against `products.db` (read-only) and returns a pandas DataFrame.
- `utils.print_html(...)` renders the DataFrame as a readable table.

In [ ]:
# Execute the generated SQL query (sql_V1) against the products.db database.
# The result is returned as a pandas DataFrame.
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# Render the DataFrame as an HTML table in the notebook.
# This makes the query output easier to read and interpret.
utils.print_html(df_sql_V1, title="Output of SQL Query V1 - ❌ Does NOT fully answer the question")

❌ **V1 output issue:** the total sales value is typically **negative**.

`qty_delta` is recorded as a negative number for sales (inventory leaving) and positive for inserts/restocks. Summing `qty_delta` (or `qty_delta * unit_price`) over sales produces a negative total — the SQL is technically valid but **semantically incorrect**.

➡️ This is why we need reflection: the model must refine its logic (e.g. `-qty_delta` or `ABS(qty_delta)` for sales) so the query reflects true sales totals.

### 3.2. Improving SQL Queries with Reflection
First the LLM reviews only the SQL text against the question and schema. Then it also considers the actual execution results, to catch subtle issues such as negative totals, missing filters or grouping errors.

#### 3.2.1. First Attempt: Refine a SQL query
**Inputs:** the user's question, the original SQL, the table schema.
**Outputs:** `feedback` (short evaluation) and `refined_sql` (unchanged if correct).
This function does **not** execute SQL — it only inspects the query text.

In [ ]:
def refine_sql(
    question: str,
    sql_query: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Reflect on whether a query's *shown output* answers the question,
    and propose an improved SQL if needed.
    Returns (feedback, refined_sql).
    """
    prompt = f"""
You are a SQL reviewer and refiner.

User asked:
{question}

Original SQL:
{sql_query}

Table Schema:
{schema}

Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
Step 2: If improvement is needed, provide a refined SQL query for SQLite.
If the original SQL is already correct, return it unchanged.

Return STRICT JSON with two fields:
{{
  "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
  "refined_sql": "<final SQL to run>"
}}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        # Fallback if model doesn't return valid JSON
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql

Run the following cell to produce the refined SQL query (V2) and compare both outputs.

In [ ]:
# Example: refine the generated SQL (V1 → V2)

feedback, sql_V2 = refine_sql(
    question=question,
    sql_query=sql_V1,   # <- comes from generate_sql() (V1)
    schema=schema, # <- we reuse the schema from section 3.1
    model="openai:gpt-4.1"
)

# Display the original prompt
utils.print_html(question, title="User Question")

# --- V1 ---
utils.print_html(sql_V1, title="Generated SQL Query (V1)")

# Execute and show V1 output
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# --- Feedback + V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

# Execute and show V2 output
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 - ❌ Does NOT fully answer the question")

❌ **As you can see...** even when the SQL text looks correct and the feedback confirms it, the results can still show a negative total. The sign inversion is a subtle semantic issue that can't always be detected by reviewing the query text alone.

👉 The agent must also reflect on the **execution output** — external feedback grounds the refinement in reality.

#### 3.2.2. Final Approach: Refine an SQL Query with External Feedback
Now the actual query results are passed to the LLM, so it can check whether the query truly answers the question.

In [ ]:
def refine_sql_external_feedback(
    question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Evaluate whether the SQL result answers the user's question and,
    if necessary, propose a refined version of the query.
    Returns (feedback, refined_sql).
    """
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {sql_query}

    SQL Output:
    {df_feedback.to_markdown(index=False)}

    Table Schema:
    {schema}

    Step 1: Briefly evaluate if the SQL output answers the user's question.
    Step 2: If the SQL could be improved, provide a refined SQL query.
    If the original SQL is already correct, return it unchanged.

    Return a strict JSON object with two fields:
    - "feedback": brief evaluation and suggestions
    - "refined_sql": the final SQL to run
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
    )


    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        # Fallback if the model does not return valid JSON:
        # use the raw content as feedback and keep the original SQL
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql

Run the following cell to see how external feedback from the query results improves SQL refinement.

In [ ]:
# Example: Refine SQL with External Feedback (V1 → V2)

# Execute the original SQL (V1)
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# Use external feedback to evaluate and refine
feedback, sql_V2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,   # V1 query
    df_feedback=df_sql_V1,    # Output of V1
    schema=schema,
    model="openai:gpt-4.1"
)

# --- V1 ---
utils.print_html(question, title="User Question")
utils.print_html(sql_V1, title="Generated SQL Query (V1)")
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# --- Feedback & V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

# Execute and display V2 results
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 (with External Feedback) - ✅ Fully answers the question")

✅ **Success!** With external feedback the sign issue is corrected (`-qty_delta` or `ABS(qty_delta)` for sales), and the output now shows the color with the highest positive total sales.

### 3.3. Putting it all together — Building the Database Query Workflow
1. Extract the database schema
2. Generate an initial (V1) SQL query
3. Execute V1
4. Reflect on V1 using execution feedback and refine the SQL
5. Execute the improved (V2) SQL query

In [ ]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = "openai:gpt-4.1",
    model_evaluation: str = "openai:gpt-4.1",
):
    """
    End-to-end workflow to generate, execute, evaluate, and refine SQL queries.

    Steps:
      1) Extract database schema
      2) Generate SQL (V1)
      3) Execute V1 → show output
      4) Reflect on V1 with execution feedback → propose refined SQL (V2)
      5) Execute V2 → show final answer
    """

    # 1) Schema
    schema = utils.get_schema(db_path)
    utils.print_html(
        schema,
        title="📘 Step 1 — Extract Database Schema"
    )

    # 2) Generate SQL (V1)
    sql_v1 = generate_sql(question, schema, model_generation)
    utils.print_html(
        sql_v1,
        title="🧠 Step 2 — Generate SQL (V1)"
    )

    # 3) Execute V1
    df_v1 = utils.execute_sql(sql_v1, db_path)
    utils.print_html(
        df_v1,
        title="🧪 Step 3 — Execute V1 (SQL Output)"
    )

    # 4) Reflect on V1 with execution feedback → refine to V2
    feedback, sql_v2 = refine_sql_external_feedback(
        question=question,
        sql_query=sql_v1,
        df_feedback=df_v1,          # external feedback: real output of V1
        schema=schema,
        model=model_evaluation,
    )
    utils.print_html(
        feedback,
        title="🧭 Step 4 — Reflect on V1 (Feedback)"
    )
    utils.print_html(
        sql_v2,
        title="🔁 Step 4 — Refined SQL (V2)"
    )

    # 5) Execute V2
    df_v2 = utils.execute_sql(sql_v2, db_path)
    utils.print_html(
        df_v2,
        title="✅ Step 5 — Execute V2 (Final Answer)"
    )

### 3.4. Run the SQL Workflow
Try different combinations of models: `openai:gpt-4o`, `openai:gpt-4.1`, `openai:gpt-4.1-mini`, `openai:gpt-3.5-turbo`.

💡 `openai:gpt-4.1` often gives the best results for self-reflection tasks.

**Important:** LLMs are stochastic, so every run may return slightly different results.

In [ ]:
run_sql_workflow(
    "products.db",
    "Which color of product has the highest total sales?",
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1"
)

## 4. Final Takeaways
- Use an LLM to turn natural-language questions into SQL queries.
- Apply reflection (with and without external feedback) to improve the generated SQL.
- Automate a complete SQL workflow, from schema extraction to query execution and refinement.
- Experiment with different models to compare performance and accuracy.

The key insight: reflection makes your agent more reliable — and **external feedback** (real execution output) catches issues the SQL text alone hides. 🎉